# What's this notebook about? 🤔

We're now looking at how we can use recurrence in neural networks in order to analyse time series data, specifically for analysis of Judo matches.

This notebook looks at both:

- Recurrent Neural Networks (RNNs)
- Long Short Term Memory networks (LSTMs)


<br>

### What we are doing today:

We'll be building a model that takes in 210 frames of a judo match and determines which of those frames contain a throw attempt, if any!

Since we're taking 210 frames in a row, we can imagine it as forming a series of positions, that's what we'll use as our input sequence to our RNN/LSTM.

<br>

**But how do we represent the frames in a judo match**

The dataset we'll be using is made up of pose estimation vectors (skeletons) that track the x,y coordinates of the keypoints of each athlete in each frame. 
These keypoints include things such as:

- ankles
- hips
- elbows

**There are 17 keypoints in total:** this means 17 xy coordinates for each player and since there are 2 players fighting, each frame will have 34 coordinates.




*If you're interested, the model used to take these skeletons from the athelets was ultralytics YOLO v11 xl*


<br>


![LSTM Diagram](https://upload.wikimedia.org/wikipedia/commons/thumb/3/3b/The_LSTM_cell.png/1200px-The_LSTM_cell.png)


<br>


### DISCLAIMER!!!

*Don't worry if you don't know anything about judo, the point of this workshop is more to get you thinking about what can be modeled as time series!*



<hr>

# The Structure of this notebook 📖

**Main goal**

Complete all of the sections labeled "Task" sequentially

<br>

**What sort of things you'll need to do for tasks**

- Some tasks will be filling in the blanks for parameters
- Others may allow a more open approach, telling you what a function should output and letting you go about implementing it whichever way you like

<br>

**Tech stack used**

- [Pytorch](https://pytorch.org/)
- [Python](https://www.python.org/)
- [Numpy](https://numpy.org/doc/)
- [YAML](https://yaml.org/)
- [Scikit-learn](https://scikit-learn.org/stable/)

<hr>

# So, what is time series? 📈

At its simplest, Time Series data is any collection of data points recorded at specific intervals over time.

Unlike a standard snapshot (like a single photo), time series data tells a story by showing how something changes.

<br>


**Most common time series application**

Stocks are a classic example of a financial time series because their value is recorded sequentially over time. In a stock market dataset, the price of a share at 10:00 AM directly influences the trading behavior at 10:01 AM.

More importantly for us, we can view stock prices as a sequence of data points taken at regular intervals.

<br>


### The 3 pillars of time series

There are 3 main pillars we must consider when looking at time series, take modeling judo using time series as an example.

To analyze Judo with AI, we look at three specific things in our data:

- **Ordering Matters:** If you shuffle the frames of a Judo throw, the story is lost. The sequence must stay in order.

- **Temporal Dependence:** What happens in frame 10 (the setup) directly influences what happens in frame 50 (the execution).

- **Sampling Rate:** In this dataset, our data is captured at a consistent rate from the YOLO v11 xl pose estimator as a constant frame rate was used, giving us a heartbeat of the match.

<br>

### Why this is hard for Normal AI 🤔

Standard Neural Networks look at each input in isolation. They have amnesia, they forget frame 1 the moment they see frame 2.

For Judo, we need an AI with a memory. This is where Recurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTM) units come in. They allow the model to keep a running summary of what it saw in the previous frames to make sense of the current one.


<hr>

# Task 0: Importing our dataset into the notebook 📂

Machine learning is useless if we don't have any data. Our first task is to load the dataset into the notebook.

For this notebook we'll be using the dataset talked about at the start of this notebook. It can be found here: [Judo Throw Interval Dataset](https://www.kaggle.com/datasets/conoroshea46/judo-throw-interval-dataset)

<br>

**Your Job:**

Import the dataset to this notebook using the 'add input' option in the right side bar

<br>


**Need help?**

Feel free to ask one of us coming around!


<hr>

# Lets see what our input data looks like 👀

In [ ]:
from IPython.display import Image, display

# Paste your copied file path here
gif_path = '/kaggle/input/datasets/conoroshea46/judo-throw-interval-dataset/judo-throw-interval-dataset/non-code-stuff/attempt_id490_annotated.gif'

display(Image(filename=gif_path, width=600))


# Task 1: Loading our dataset into a useful format the program

Our data is also not much use if it's just sitting in a folder, lets get it into the actual code.

If you look at the dataset you'll see that we have 3 .npy files:

- input_data.npy
- labels.npy
- attempt_ids.npy

The only 2 we need to worry about right now are input_data.npy and labels.npy since they contain they contain the input sequences and the labels for those input sequences respectively


<br>

### What are .npy files? 🤔

.npy is the file extension for numpy. 

Numpy is just a library in python that provides classic style arrays (continous blocks of allocated addresses) and some useful functionality with them.

We're interested in them for machine learning as due to the way they are stored in memory, they work pretty seamlessly with pytorch tensors.


*See documentation for numpy here: https://numpy.org/doc/stable/*


## Your task 👷‍♂️

Your first coding job is to store the numpy arrays:

- input_data.npy
- labels.npy

as variables in python.


*Some of the code has already been done to help you. You just need to fill in the sections marked with a ...*


In [ ]:
import numpy as np


input_data_path = '/kaggle/input/datasets/conoroshea46/judo-throw-interval-dataset/judo-throw-interval-dataset/throw-locator-model-data-final/input_data.npy'
data_labels_path = '/kaggle/input/datasets/conoroshea46/judo-throw-interval-dataset/judo-throw-interval-dataset/throw-locator-model-data-final/labels.npy'

input_data = np.load(input_data_path)
label_data = np.load(...)



# don't worry about these 2 lines
attempt_ids_path = '/kaggle/input/datasets/conoroshea46/judo-throw-interval-dataset/judo-throw-interval-dataset/throw-locator-model-data-final/attempt_ids.npy'
attempt_ids = np.load(attempt_ids_path, allow_pickle=True)

---

# Task 2: Lets have a look at the shape of our input data 👀

Now that we've loaded our data as numpy arrays we can start to work with, and understand it.

For our model, we need to know the dimensions of our input data, this includes knowing:

- How many features in each frame vector
- How many vectors in a sequence

Since our data is stored in numpy arrays we can use the shape function provided within numpy to check the dimensionality of our arrays.

The shape function returns a tuple that's size varies on on how many we dimensions we have in the numpy array. e.g. if we have 3 dimensions we'll get a tuple of size 3.

<br>

**What would the elements in a 3d tuple be?**

(Samples, Time Steps, Features)

<br>

**Well, how many dimensions do we have?**

For our input_data array, since we have:

- 2481 training points in the dataset
- sequences of 210 frames
- 68 features per frame

Our input_data array must be 3d


Our labels array will still be a 3d array since it must have the same number of dimensions as our *,however,* one of the values in that shape tuple will be different.

<br>

**Why must the labels array have the same number of dimensions as the input_data array?**

We'll look at the dimensions individually:

- **Num of samples:** This is trivial, every input seqeunce must have a corresponding label
- **Num of time steps:** Since we are predicting for each frame, if the frame contains a throw, we need a label for each frame - this results in 210 time steps
- **Features:** This isn't directly tied to the input array dimensions, however, if we have no output features, our model isn't actually predicting anything so there would be no point.


However, more importantly if our model outputs an array of a certain size, our label array must match that size.

<br>

#### A closer look at the number of features in the label array

The number of features in our label array actually coresponds to the number of things we're predicting from the data.

e.g. If we had data on stocks, we could get the model to predict 3 things for every sequence:

- Close price
- Moving Averages
- Volume

This would be 3 features.

<br>

**What about our judo prediction model?**

Since we are only predicting 1 thing, i.e. whether a frame contains a throw or not, we only have 1 feature in our labels array

<br>

## Your task

Use the numpy shape function to verify that your input data and label data tensors match our desired dimensions.


*Hint: If done correctly, you should see the tuples (2481, 210, 68) and (2481, 210, 1)*


<br>



**Want to know more about the data?**

[Check out the dataset description here!](https://www.kaggle.com/datasets/conoroshea46/judo-throw-interval-dataset)


In [ ]:
input_data_shape = ... # hint: name_of_numpy_array.shape
label_data_shape = ...

print(f"Shape of input_labels array:  {input_data_shape}")
print(f"Shape of labels array:  {label_data_shape}")

<hr>

# Loading our hyperparameters ⚙️

We'll use a file format called YAML to store our hyperparameters, you'll find this already in your input directory.

<br>

**Why YAML?** 

In machine learning, we often need to tweak dozens of variables (learning rate, weights, batch size). Hard-coding these inside your Python scripts is bad practice because:

- Decoupling: You can change how the model trains without ever touching the actual code.

- Reproducibility: You can save the YAML file alongside your trained model weights so you know exactly which settings produced that specific result.

- Readability: YAML is designed to be "human-readable," making it much easier to scan than a wall of nested JSON.

<br>

**What does a YAML file look like? 📜**

YAML relies on indentation to represent data

The one in this notebook looks like:

In [ ]:
!cat /kaggle/input/datasets/conoroshea46/judo-throw-interval-dataset/judo-throw-interval-dataset/config.yaml

### Don't worry about the implementation of how we read the file but if you're interested it's here:

In [ ]:
import yaml

# parse the yaml file
def load_config(config_path="/kaggle/input/datasets/conoroshea46/judo-throw-interval-dataset/judo-throw-interval-dataset/config.yaml"):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config

# you can now access model configurations like a dictionary
model_configurations = load_config()

---

# Task 3: Split your datasets into training and test data 📂

Whenever we're doing machine learning, we usually want to split our dataset into 3 subsets:

- **Training**- Used for teaching the model 
- **Evaluation** - Used for seeing how well the model is doing *while* it's training
- **Testing** - Used for seeing how good a model we have after all training is complete

Evaluation data is useful for seeing if our model is learning the relationships in the data properly, instead of just memorizing the training data. If we can detect that it's not learning properly we can stop the training loop early and tweak the components in the training loop - this saves us time that would've been wasted in letting the loop complete.


<br>

### How do we split the data between the 3

We must make it so the split is disjoint and the majority of the data is allocated as training data.

Since our dataset is not tiny but still not very big, a common split is:

- **Training:** 80%
- **Testing:** 10%
- **Validation:** 10%

*We've defined this split in the config.yaml file given*


<br>

## Your Task

Your job is to split the input_data and label_data numpy arrays into 6 new numpy arrays: 3 each for the testing, training and validation data of the input_data and label_data.



<br>


### How do we go about coding this split

Scikit-learn provides a function train_test_split() for splitting a given numpy array into 2 (note: it can also split some other data types!).

*However,* we need to split our dataset into 3.

Therefore you must first:

1. Split the dataset 80/20, 80 to training and 20 to be SHARED BETWEEN testing and validation.

***Then,***

2. Split the 20% that we shared between the testing and validation data in 2.


This leaves us with our desired split of 80|10|10

<br>

***IMPORTANT 🚨🚨🚨***

You MUST ensure that the indices of the data points from the input data match up with the indices of their labels. 

If you don't do this you'll lose track of which label is for which data point and if that happens, your training loop is useless.


The train_test_split() function allows us to avoid this issue by letting us set the random state, and set that as the same random state for ALL created numpy arrays.

In [ ]:
from sklearn.model_selection import train_test_split

train_in, rest_in, train_lab, rest_lab, train_ids, rest_ids = train_test_split(
    ...,  # hint: you need to remember to keep your input data and their labels alligned. What data may be our input here?
    ..., 
    attempt_ids,
    test_size=model_configurations['data']['split']['val_test_ratio'],  # this will be 0.8, look at the yaml output from earlier to see why!
    random_state=model_configurations['data']['split']['random_state']
)

# now split that 20% exactly in half (50/50)
# Result: 10% Eval, 10% Test
# eval_in, test_in, eval_lab, test_lab will all be numpy arrays
val_in, test_in, val_lab, test_lab, val_ids, test_ids = train_test_split(
    rest_in, 
    rest_lab, 
    rest_ids,
    test_size=model_configurations['data']['split']['eval_test_split'], 
    random_state=model_configurations['data']['split']['random_state']
)

---

# Task 4: Loading our dataset arrays into containers that work with pytorch's framework 🚚

We currently have 6 numpy arrays that, together, represent our dataset.

However, if we want to be able to utilze pytorch's framework to make our training loop easier, we need to condense this down to 3 Pytorch [Dataloader](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) objects.

One for each of:

- Training data
- Testing data
- Validation data

<br>

### How do we create these Dataloader objects

We first need to define our own special class that uses the pytorch framework.

Every instance of this class will contain 2 tensors.

- Input data  (converted from numpy array)
- Labels  (converted from numpy array)

This should make it so that:

- The training data object contains its input data and labels
- Same idea for the testing data object
- Same idea for the validation data



<br>

### More on the special class for this project

Due to judo being hard to model with pose estimation consitently, some of the frames in our input data are bad - this can be from the model used to track the skeletons losing sight of the skeletons for a couple frames.

However, if we want to be able to use [batching]((https://github.com/EdinburghAI/workshops/blob/main/TheoreticalWorkshops/Sem2Workshop1/yolo-workshop-api/Additional-notes/Batching.md)) in our training loop later, every sequence must have the same number of time steps (in our case 210).

Therefore, the data has been encoded with sentinel values which are set to 99. This is so that we can flag to our optimizer to not punish the model for a bad prediction on those frames.

The issue with this approach is that leaving the value of 99 in will result in our multiplications (in our model's layers) giving massive values. This s bad because these big weight changes will completely dwarf any fine tuned changes made by our good training examples.

The solution is to replace an 99 values with 0s, so they don't blow up the gradients in the model.

We do this using a frame wise mask of 1s and 0s, if we multiply 99 by zero, it will be 0. 

***Note though: This is only for use in our training loop, we don't get rid of the 99s in the input array object***

<br>

## Your Task

We have a few things to do here (some have been done for you already:

1. Define our special class
2. Create the mask needed for the the sentinel value handling and have our special class store it
3. Make our special class also store the input data and labels numpy arrays as tensors so that they're ready for use later


<br>

**If you need help with understanding the masking just ask!***


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset


class JudoDataset(Dataset):
    def __init__(self, input_data, ..., data_ids):  # hint: what should each special dataset instance contain?
        self.inputs = input_data.copy() 
        self.labels = data_labels.copy()
        self.ids = data_ids.copy() 
        
        # creating mask so the loss function later knows which frames not to penalize
        # don't penalize any features that contain the sentinel value of 99 which represents player not detected
        # the mask is made up of 1s and 0s since if we multiply by 0, we get 0, meaning we can easily get 0 loss for frames which has sentinel values
        self.masks = (self.inputs[:, :, 0:1] != 99.0).astype(np.float32)
        
        # now replace the 99s with 0.0 so the LSTM doesn't multiply by 99 and explode its gradients.
        # we still have a variable tracking where the 99s are though since we made the mask above
        self.inputs[self.inputs == 99.0] = 0.0
        
        # convert input numpy arrays to tensors
        self.inputs = ...  # hint:  dtype=torch.float32
        self.labels = ...
        self.masks = ...



Pytorch also requires you to define functions in your special class for :

- getting the number of sequences in your input and labels array
- returning the next item in your input array (for training)


This is purely because the Dataloader objects we'll create using our special class require these functions to exist.

Don't worry about implementing these in this workshop, it's been done for you here!


In [ ]:
def get_length(self):
    return self.inputs.shape[0]
    

# need to return masks here so the data loader instance gets them
def get_item(self, index):
    return self.inputs[index], self.labels[index], self.masks[index], self.ids[index]

JudoDataset.__len__ = get_length
JudoDataset.__getitem__ = get_item

# We can now create our Dataloader objects using our special class 🛠️

In [ ]:
from torch.utils.data import DataLoader

# instantiate split dataset into their own dataset objects
train_dataset = JudoDataset(train_in, train_lab, train_ids)
test_dataset = JudoDataset(test_in, test_lab, test_ids)
val_dataset = JudoDataset(val_in, val_lab, val_ids)

# move to dataloader obejcts for use in training loop
training_dataset_loader = DataLoader(train_dataset, batch_size=model_configurations['train']['batch_size'], shuffle=True)
testing_dataset_loader = DataLoader(test_dataset, batch_size=model_configurations['train']['batch_size'], shuffle=True)
val_dataset_loader = DataLoader(val_dataset, batch_size=model_configurations['train']['batch_size'], shuffle=True)



---

# Task 5: Designing our Loss function 📉

We want to loss function to punish our model for mislabeling whether there is a throw attempt in a given frame or not.

This means that we punish:

- Saying that there is a throw attempt in a frame when there's not
- Saying that there's not a throw attempt in a frame when there actually is


***Recall: We want our model to predict a 1 in that frames element if there is a throw, and a zero if there's not. Since our model outputs values between 0 and 1 per frame, we can actually view each value as the probability that the frame contains a throw attempt***

<br>

## Considering our dataset

Since we're doing binary classification, when considering our loss function, we need to consider our data is split in terms of:

- Frames with throws
- Frames without throws

In our dataset the split is:

- Frames with throws = 16465
- Frames without throws = 504545

<br>

### Why is this important? 🤔


Remember that we're doing binary classification and since there are many more frames without throws than there are frames with throws, if our model just predicts that there isnt a throw every time it will actually have a 97% success rate... even though it's no actually learned anything and will hence perform terribly if it was tested on a dataset where that are more frames with throws than frames without throws.

<br>

#### How do we solve this issue?

We need to weight our loss function so that it punishes labeling frames that should have throws wrong more harshly.

We do this by simply scaling our loss when this is the case.

<br>

## Bringing it all together, what do we actually need our loss function to do?

- Punish our model for mislabeling whether there is a throw attempt in a given frame or not.
- Punish our model more harshly if it labels frames that should have throws wrong.
- **Most importantly**: punish prediction by frame, not by the sequence as a whole


<br>

### How do we implement this: BCE Loss (Binary Cross-Entropy Loss)

You can think of BCE Loss as a measure of surprise - If the model sees a throw happening but predicted it wouldn't, it is very "surprised," and the loss value spikes.

For a single frame, the formula is:

$L=−(ylog(p)+(1−y)log(1−p))$

**Where:**

- y is the True Label (0 or 1).

- p is the Predicted Probability (a value between 0 and 1).

<br>


#### Why not just use mean squared error?

Unlike MSE, which is quite gentle with small mistakes, BCE Loss is much more aggressive. Because it uses logarithms, it punishes the model exponentially more the further it gets from the correct answer. This native intensity makes it much better at forcing the model to distinguish between a throw and a non-throw.


<br>

#### Implementing BCE Loss in torch 

Pytorch actually provides functionality of BCE Loss!

However, we're specifically going to use BCEWithLogitsLoss as it will internally apply the sigmoid activation function (which is the thing that makes our model output probabilities as opposed to any number in the range $(-\infty, \infty)$).

BCEWithLogitsLoss also applies the weight we were talking about earlier for us (if we pass the weigh as an argument when we instantiate the object.


<br>

#### What is a Logit?

A logit is the raw, unscaled output from your model's last linear layer. It can be any number from negative infinity to positive infinity.

To get a probability (0 to 1), we usually pass that logit through a Sigmoid function.

- BCELoss: Expects you to apply the Sigmoid yourself first.

- BCEWithLogitsLoss: Takes the raw number and does the Sigmoid calculation internally.

*This basically just means that we don't have to put a sigmoid layer at the end of our models since we're putting it in our loss function instead*


<br>


***Where the 97% figure came from***

$$\text{Accuracy} = \left( \frac{\text{Frames Without Throws}}{\text{Total Frames}} \right) \times 100$$

$$\text{Accuracy} = \left( \frac{504,545}{504,545 + 16,465} \right) \times 100$$

$$\text{Accuracy} = \left( \frac{504,545}{521,010} \right) \times 100 \approx 96.84\%$$

<br>

## Your Task 💻

Implement a loss function class that implements BCE loss.

Do this by filling in the ... sections in the code below!


*Note: This code implements the masking mentioned earlier but don't worry about it as it has been applied for you. If you're curious about the masking just give me a shout!*



In [ ]:
import torch.nn as nn
import torch



class MaskedJudoLoss(nn.Module):
    def __init__(self, pos_weight_val=30.64):
        super().__init__()

        # init tensor storing weight for how we punish predicting no throw attempts vs throw attemps
        # since there's a lot more frames without throws that with throws in our dataset, we need to punish predicting no throw when 
        # there is a throw more harshly
        # register_buffer ensures this tensor moves to the GPU automatically with the model
        self.register_buffer('pos_weight', torch.tensor([pos_weight_val], dtype=torch.float32))
        
        # init var storing the BCE loss function
        self.bce = nn.BCEWithLogitsLoss(reduction='none', pos_weight=self.pos_weight)



    def forward(self, predictions, targets, masks):

        if self.bce.pos_weight.device != predictions.device:
            self.bce.pos_weight = self.bce.pos_weight.to(predictions.device)


        # get raw error for each item in batch
        # raw_loss will be a tensor containing the loss for each item in the batch (think of it as a list containing the computed losses of all points in the batch)
        raw_loss = self.bce(predictions, ...)   # hint: if we have what our model predicted, what other data do we need to compute the loss?

        # apply our mask so we're not punishing the model for predictions of frames that had a sentinel value
        # remember at this stage, raw_loss is still a tensor so we're scaling the elements in this tensor element wise
        masked_loss = raw_loss * masks


        # now get the average loss for the batch
        # remember from the yolo workshop: We add 1e-8 to the denominator just as a safety net to prevent dividing by exactly zero
        ... = masked_loss.sum() / (masks.sum() + 1e-8)   # hint: What is this the equation for?

        return final_loss

# We'll also now create our criterion so we can use it to apply our loss function later

## What is criterion 🤔

Here we're initializing a variable called criterion (don't worry you don't have to implement this code yourself, it's already done for you).

criterion is essentially the object we'll use in our training loop for calculating the loss for a given batch in our training data - it's basically just a container for our loss function, making our training loop syntax simpler.

<br>

**Why the name?**

The literal definition of criterion in English is: "a standard or a rule by which something is judged".

In Deep Learning, that standard is our Loss.



In [ ]:
# here for playing around with the weight
#model_configurations['train']['pos_weight'] = 0.32

criterion = MaskedJudoLoss(model_configurations['train']['pos_weight'])

---

# Task 6: It's finally time to build the our recurrent models 🏗️

We're first going to implement an RNN as a benchmark to see just how much better LSTMs are than them!

Pytorch has actually made the syntax stupidly simple for implementing these models, however, since our RNN and LSTM layers will just output the raw information they've learned we're also going to have a couple of feed forward layes after them so the model can almost bring together the information that the RNN or LSTM has learned.

Think of our model in two parts:

- The Backbone (RNN/LSTM): This is the "brain" that looks at the 210 frames of Judo and tries to remember the movement patterns. It outputs a complex vector of numbers for every frame.
- The Head (Linear Layers): This is the decision maker. It takes the complex information from the backbone and compresses it down into a single number: The probability that the frame contains a throw.

<br>

### Pytorch Documentation

- [RNN](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html)
- [LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)


<br>


### Stuck?

Some of the comments in the task 7.2 LSTM code might help you! 

# Task 6.1: Define the RNN

We're gonna start with a 2 layer RNN as it has just enough parameters for the model to learn deep relationships, withoutout overtraining due to the small dataset size.

Feel free to experiment with it though!

<br>


### What arguments does RNN expect?

- **input_size:** The number of expected features in the input sequence. In this project, it is set to 68 because there are 17 keypoints per player, 2 coordinates per keypoint (x, y), and 2 players in the match.
  
- **hidden_size:** The number of features in the internal hidden state. This acts as the model's "memory capacity" to track patterns across the 210 frames.

- **num_layers:** The number of recurrent layers to stack on top of each other. Using 2 layers allows the model to learn deeper, more complex relationships in the movement data.

- **dropout:** A regularization technique that randomly "turns off" a percentage of neurons during training to prevent the model from just memorizing the data.

- **batch_first:** A boolean that, when set to True, tells the model the input and output tensors are provided in the format (batch, seq, feature). This matches the shape of your input_data array.


<br>

## Your task

Your task here is just to fill in the sections marked with ...


In [ ]:
import torch.nn as nn

# use the gpu if it supports cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ThrowLocatorRNN(nn.Module):
    def __init__(self, input_size=68, hidden_size=128, num_layers=2, dropout_rate=0.3):
        super().__init__()

        # define a variable storing our RNN layer
        self.rnn = nn.RNN(input_size=input_size,  # hint: what arguments are missing that we said we needed above?
                          hidden_size=hidden_size,
                          ...,
                          dropout=dropout_rate if num_layers > 1 else 0,
                          batch_first=...
                          )
        
        # Intermediate linear layer to process RNN features
        self.fc1 = nn.Linear(..., 64)  # hint: how many features are in the final hidden state vector outputted by the final rnn layer?
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

        # Final output layer
        self.fc_out = nn.Linear(64, ...)  # hint: how many features are we predicting per sequence?


    def forward(self, input_sequence):

        # Unlike LSTM which returns (output, (h_n, c_n)), 
        # a basic RNN only returns (output, h_n).
        # We still use the underscore to discard the final hidden state (h_n).
        rnn_output, _ = self....  # hint: what is the first layer in the model we pass the input through?

        # Pass the RNN features through the rest of the architecture
        fc1_output = self.fc1(rnn_output)
        activation_output = self.relu(fc1_output)
        dropout_output = self.dropout(activation_output)

        # Final logits [Batch, 210, 1]
        logits = self.fc_out(dropout_output)

        return logits


# instantiate an rnn model from the class
rnn_model = ThrowLocatorRNN()
rnn_model.to(device) # move the model to the gpu


# Task 6.2: Define the LSTM

We're gonna start with a 2 layer LSTM as well so that we're giving a fair comparison to the RNN.

Feel free to experiment with it though!

<br>

### What arguments does LSTM expect?

- **input_size**: The number of expected features in the input sequence. In this project, it is set to 68 because there are 17 keypoints per player, 2 coordinates per keypoint (x,y), and 2 players in the match.

- **hidden_size:** The number of features in the internal hidden state. This acts as the model's "memory capacity" to track patterns across the 210 frames.

- **num_layers:** The number of recurrent layers to stack on top of each other. Using 2 layers allows the model to learn deeper, more complex relationships in the movement data.

- **dropout:** A regularization technique that randomly "turns off" a percentage of neurons during training to prevent the model from just memorizing the training data.

- **batch_first:** A boolean that, when set to True, tells the model the input and output tensors are provided in the format (batch, seq, feature). This ensures the model knows the first dimension in our input tensor is the batch dimension, matching our input_data array.

<br>

### Your Task

Once again your task is to fill in the ... sections, but also notice just how similar the syntax is to RNNs



In [ ]:
import torch.nn as nn


class ThrowLocatorModel(nn.Module):
    def __init__(self, input_size=68, hidden_size=128, num_layers=2, dropout_rate=0.3):
        super().__init__()

        
        # define a variable storing the LSTM layers in the model
        self.lstm = nn.LSTM(input_size=input_size,
                            ...,  # hint: what argument is missing that we said we needed?
                            num_layers=num_layers,
                            dropout=dropout_rate if num_layers > 1 else 0,
                            batch_first=True
                            )
        
        # we need batch_first = True so that the model knows the first dimension in our input tensor the the batch dimension
        
        
        # we now want a linear layer so that the lstm can process its findings in 2 stages as opposed to straight to output
        # the choice of 64 is somewhat arbitrary but it's common to go for powers of 2
        self.fc1 = nn.Linear(hidden_size, 64)  # fc stands for fully connected (since linear layers are fully connected)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)

        # now compress the output to 1 feature per frame (since we're only predicting 1 thing)
        # we're predicting the confidence that that frame contains part of a throw
        # we do this using one more linear layer
        self.fc_out = nn.Linear(64, ...)  # hint: how many features are we predicting per sequence?


    def forward(self,input_sequence):

        # first put it through the lstm
        # lstm outputs both the final hidden and cell states, but we only care about the output vector
        # therefore we need to unpack as a tuple but discard the cell state
        # the tuple output of the lstm is (output, (h_n, c_n))
        lstm_output, _ = self.lstm(input_sequence)

        # then through the first linear layer
        fc1_output = self.fc1(...)  # hint: what is the input for the first feed forward layer? (This was mentioned in a text chunk above)

        # then through the activation function (relu)
        activation_output = self.relu(fc1_output)

        # now through our dropout layer
        dropout_output = self.dropout(activation_output)

        # finally put it through the last linear layer to get our our output
        logits = self.fc_out(dropout_output)

        return logits


# instantiate an lstm model from the class 
lstm_model = ThrowLocatorModel()
lstm_model.to(device) # move the model to the gpu


---


# Task 7: Choose your Training Strategy 🎛️

When we're training a neural network we need a rule for how we update our weights.

<br>

**Is our loss function not the rule?**

Not quite

We use the loss function as part of the equation for calculating our weight update. It just tells us whether the weights in the given layer should go up or down and vaguely by how much.

<br>

**Where optimizers come into this**

We can use what's called an optimizer for defining and applying our weight update rule in our training loop.

A couple of common options are:

- Stochastic Gradient Descent (we saw this last semester!) -
- ADAM


<br>

For this workshop we'll use Adam as our optimizer because it’s simple to use and works well in most situations:

- **It automatically adjusts itself** - 
Instead of using the same step size everywhere, Adam changes how big each step is for different parts of the model. Think of it like cruise control that slows down when the road gets twisty and speeds up when things are clear.

- **It remembers past progress** - 
Adam keeps track of which direction things were improving before and continues moving that way. Like a ball rolling downhill, it doesn’t stop at every small bump, it keeps going toward a better solution.

- **It’s very forgiving** - 
Adam usually works well even if you don’t pick the perfect settings. For beginners, this is great because you can get good results without a lot of trial and error.


<br>

## Your Task: What you've got to do

*Have a look at the ADAM documentation here: https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html*

Decide what order the learning_rate and weight_decay parameters should go into the Adam constructor function in the code chunk below.


<br>


***Still confused about optimzers?***

[Here's a bit more information](https://github.com/EdinburghAI/workshops/blob/main/TheoreticalWorkshops/Sem2Workshop1/yolo-workshop-api/Additional-notes/Glossary.md)



In [ ]:
import torch.optim as optim

rnn_params = rnn_model.parameters()

rnn_optimizer = optim.Adam(
    rnn_params, 
    lr=model_configurations['train'][...],  # hint: what does "lr" likely stand for?
    weight_decay=model_configurations['train']['weight_decay']
)


lstm_params = lstm_model.parameters()

lstm_optimizer = optim.Adam(
    ...,  # hint: if we passed the rnn parameters to the rnn optimizer, what model parameters should we pass to the lstm model? 
    lr=model_configurations['train']['learning_rate'],
    weight_decay=model_configurations['train']['weight_decay']
)

---

# Task 8: Lets train our models ⚡

Here is where implement the Backpropagation Through Time (BPTT) that we talked about in the workshop slides!

<br>

We'll define our 
There are essentially 3 stages in our training loop:

1. Split the dataset into batches for the current epoch

Switch the model into training mode

*Start looping over the batches (nested loop)*

    2. Run through the training data batch, updating the weights as you go

Switch the model out of training mode and into evaluation mode - this means no more weight updates when the model is in eval mode instead of training mode

    3. Run through the validation data batch, **NO WEIGHT UPDATES**



<br>

### Stage 2: looping over the training data

At its core, this is basically a 4 step process:

1. Run the inputs in the TRAINING batch through the model to get the outputs tensor for the TRAINING batch
2. Compute the total loss for the batch
3. Run backpropagtion to compute gradients
4. Run the optimizer to actually update the weights in the network
5. Do this for all batches in the TRAINING data


<br>


### Stage 3: looping over the validation data

Since the evaluation data is not for training the network, rather viewing how it's doing as it trains, it's very important that you first switch the model OUT of training mode

After that the basic idea is:

1. Run inputs in the VALIDATION batch through the model to get the outputs tensor for the VALIDATION batch
2. Compute some metric to check how the model is doing - commonly just the loss
3. Record the metric
4. Do this for all batches in the VALIDATION data


<br>

#### What is the torch function for backpropagation?

[backwards()](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

***Note: backwards() is very adaptible, it will automatically apply backpropagation through time for us!***

<br>

#### Some guidance on training loops in PyTorch**

- [Training Loops](https://github.com/EdinburghAI/workshops/blob/main/TheoreticalWorkshops/Sem2Workshop1/yolo-workshop-api/Additional-notes/Training%20Loops%20in%20PyTorch.md)

- [How does Pytorch implement backpropagation?](https://github.com/EdinburghAI/workshops/blob/main/TheoreticalWorkshops/Sem2Workshop1/yolo-workshop-api/Additional-notes/Computational%20Training%20Graphs.md)

<br>

#### Batching

[What is batching?](https://github.com/EdinburghAI/workshops/blob/main/TheoreticalWorkshops/Sem2Workshop1/yolo-workshop-api/Additional-notes/Batching.md)




In [ ]:
# this is here in case you want to play about with the number of epochs
model_configurations['train']['epochs'] = 90

# Task 8.1: Train the RNN

We'll first create an RNN from the blueprint we defined earlier and see how it does.

<br>

### Your job

Fill in the sections marked with ...

In [ ]:
from tqdm import tqdm


num_epochs = model_configurations['train']['epochs']


train_losses_rnn = []
eval_losses_rnn = []

# now for the actual training loop
# since now have eval data, training loop splits into 2 phases
for epoch in range(num_epochs):

    # move model to training mode
    rnn_model.train()  # Activates Dropout and BatchNorm layers
    train_epoch_loss = 0.0

    # this line is just to give a progress bar for how far through the dataset you are as the model trains, it has no effect on the model itself
    train_pbar = tqdm(training_dataset_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)
    for inputs, targets, masks, batch_ids in train_pbar:
        inputs, targets, masks = inputs.to(device), targets.to(device), masks.to(device)

        rnn_optimizer.zero_grad()

        outputs = rnn_model(...)  # hint: what must be passed to the model in order to get outputs?


        # calculate loss for batch
        loss = criterion(outputs, , masks)  # hint: if we have the outputs, what other vector is needed to calculate the loss?

        # now for backpropagation and weight updates
        loss.backward()
        rnn_optimizer.step()


        # update the running total for loss of the epoch
        train_epoch_loss += loss.item()
        train_pbar.set_postfix(loss=loss.item())



    # now for the eval phase

    # first switch the model to eval mode
    rnn_model.eval()
    eval_epoch_loss = 0.0

    eval_pbar = tqdm(val_dataset_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Eval]", leave=False)
    with torch.no_grad():
        for inputs, targets, masks, batch_ids in eval_pbar:
            inputs, targets, masks = inputs.to(device), targets.to(device), masks.to(device)
            
            ... = rnn_model(inputs)  # hint: what's the first step of the validation phase of the training loop?
            loss = ...(outputs, targets, masks)  # hint: what object stores our loss function?
            eval_epoch_loss += loss.item()


    # --- END OF EPOCH SUMMARY ---
    avg_train_loss = train_epoch_loss / len(training_dataset_loader)
    avg_eval_loss = eval_epoch_loss / len(val_dataset_loader)
    
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {avg_train_loss:.4f} | Eval Loss: {avg_eval_loss:.4f}")
    print("-" * 30)

    train_losses_rnn.append(avg_train_loss)
    eval_losses_rnn.append(avg_eval_loss)

# Task 8.2: Train the LSTM

We'll now create an LSTM to see if the architecture change actually helps.

<br>


### Your Job

Fill in the sections marked ...


In [ ]:
from tqdm import tqdm

lstm_model.to(device) # move the model to the gpu


num_epochs = model_configurations['train']['epochs']


train_losses_lstm = []
eval_losses_lstm = []

# now for the actual training loop
# since now have eval data, training loop splits into 2 phases
for epoch in range(num_epochs):

    # move model to training mode
    lstm_model.train()  # Activates Dropout and BatchNorm layers
    train_epoch_loss = 0.0

    # this line is just to give a progress bar for how far through the dataset you are as the model trains, it has no effect on the model itself
    train_pbar = tqdm(training_dataset_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)
    for inputs, targets, masks, batch_ids in train_pbar:
        inputs, targets, masks = inputs.to(device), targets.to(device), masks.to(device)

        lstm_optimizer.zero_grad()

        outputs = lstm_model(...)  # hint: what must be passed to the model in order to get outputs?


        # calculate loss for batch
        loss = criterion(outputs, , masks)  # hint: if we have the outputs, what other vector is needed to calculate the loss?

        # now for backpropagation and weight updates
        loss. ...  # hint: what's the function for back propagation?
        lstm_optimizer.step()


        # update the running total for loss of the epoch
        train_epoch_loss += loss.item()
        train_pbar.set_postfix(loss=loss.item())



    # now for the eval phase

    # first switch the model to eval mode
    lstm_model.eval()
    eval_epoch_loss = 0.0

    eval_pbar = tqdm(val_dataset_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Eval]", leave=False)
    with torch.no_grad():
        for inputs, targets, masks, batch_ids in eval_pbar:
            inputs, targets, masks = inputs.to(device), targets.to(device), masks.to(device)
            
            outputs = lstm_model(inputs)
            loss = criterion(outputs, targets, masks)
            eval_epoch_loss += loss.item()


    # --- END OF EPOCH SUMMARY ---
    avg_train_loss = train_epoch_loss / len(training_dataset_loader)
    avg_eval_loss = eval_epoch_loss / len(val_dataset_loader)
    
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {avg_train_loss:.4f} | Eval Loss: {avg_eval_loss:.4f}")
    print("-" * 30)

    train_losses_lstm.append(avg_train_loss)
    eval_losses_lstm.append(avg_eval_loss)

---

# Task 9:  Evaluate your models 🧪

Now it's time to see which model performed better!

<br>

### How do we evaluate the model?

We'll use F1 metrics to evaluate the model as they give they allow us to track:

- **Precision:** how often is the model correct when it predicts that a frame contains a throw?
- **Recall:** how often does it predict there is a throw?


**Precision computation**

$$\text{Precision} = \frac{\text{True Positives}}{\text{True Positives} + \text{False Positives}}$$

**Recall computation**

$$\text{Recall} = \frac{\text{True Positives}}{\text{True Positives} + \text{False Negatives}}$$

**F1 Metric computation**

The F1 Score is the harmonic mean of the two. It balances the quality (Precision) with the quantity (Recall).

<br>

#### What do we mean by true positives etc?

- **True Positive (TP):** the model predicted that a frame contains a throw and it actually does
- **True Negative (TN):** the model predicted that a frame DOES NOT contain a throw and it actually DOES NOT
- **False Positive (FP):** the model predicted that a frame contains a throw and it actually DOES NOT
- **False Negative (FN):** the model predicted that a frame DOES NOT contain a throw and it actually DOES 



<br>

### What do we classify as the model predicting that a frame contains a throw?

In a perfect world, our model would always predict a probability of 1 for frames that contain a throw.

However, we do not live in a perfect world...

Therefore, we need to define a threshold value where we say "If the output value is greater than or equal to this value, the model predicted that the frame contains a throw attempt"

This threshold has been defined in the config.yaml file as 0.7.

0.7 is generally a bit of a sweet spot for this sort of thing given our dataset size, not too harsh but also not too relaxed.


<br>


### But what does a good model look like in our case?

Due to the nature of small computer vision datasets, even a 60% F1 score is considered decent for our dataset size.

The main factors that hold us back are:

- **Dataset size:** there are lots of ways to throw in judo, our dataset does not capture enough examples of each throw to be professional level
- **Judo is a highly complex sport technique wise:** the players may be moving around quickly together, but this isn't necessarily always because one player is attempting a throw
- **Pose estimation can be un-reliable:** since judo is such a fast moving sport, pose estimation software will sometimes glitch and lose track of a player

And we just have to work with what we have since computer vision data is very time-consuming to gather **(for context: this dataset took about 50 hours to prepare!)**


<br>



In [ ]:
# here for playing around with the confidence threshold
model_configurations['inference']['threshold'] = 0.7

# Task 9.1: Evaluate the RNN


### Your job

Fill in the sections in the code below marked with ...

In [ ]:
# now test the model (inference)

total_val_loss = 0.0

rnn_model.eval()


# init metrics for F1 score testing
tp = 0
fp = 0
tn = 0
fn = 0

actual_num_frames_with_throws = 0

throw_predicted_threshold = model_configurations['inference']['threshold']

successful_throw_ids = []

test_pbar = tqdm(testing_dataset_loader, desc="Testing Model", leave=True)
with torch.no_grad():
    for inputs, targets, masks, batch_ids in test_pbar:
        inputs, targets, masks = inputs.to(device), targets.to(device), masks.to(device)
        
        # run inference on the inputs
        logits = rnn_model(inputs)
        
        # turn outputs into probabilities of each frame containing a throw
        probs = torch.sigmoid(...)   # hint: the raw scores of the model get passed to the sigmoid activation function. What are they called?



        predictions = (probs >= ...).float()  # hint: if the probability predicted by the model is greater than or equal to what? (for classifying as throw attempt)

        # Only look at real frames (where mask == 1)
        # We flatten them to make the comparison easy
        active_preds = predictions[masks == 1]
        active_targets = targets[masks == 1]

        # Update counts
        tp += ((active_preds == 1) & (active_targets == 1)).sum().item()
        fp += ((active_preds == 1) & (active_targets == 0)).sum().item()
        fn += ((active_preds == 0) & (active_targets == 1)).sum().item()
        tn += ((active_preds == 0) & (active_targets == 0)).sum().item()
        

        # since only 1s and zeros, can just sum to get num frames with throws
        actual_num_frames_with_throws += active_targets.sum().item()
                
        for i in range(len(predictions)):
            if (predictions[i] == 1).any() and (targets[i] == 1).any():
                successful_throw_ids.append(batch_ids[i])
        

precision = tp/(tp + fp + 1e-8)   # Note: we have the + 1e-8 to prevent division by 0

recall = tp/(actual_num_frames_with_throws + 1e-8)

# hint: what is the equation for f1 score?
f1_score = 2* (... * recall)/(... + ... + 1e-8)   # Note: we have the + 1e-8 to prevent division by 0

print("----------Testing Results for RNN----------")
print(f"Total num of frames with throws in Dataset: {actual_num_frames_with_throws}")
print(f"True Positives (Hits):   {tp}")
print(f"True Negatives:   {tn}")
print(f"False Positives (Gaps):  {fp}")
print(f"False Negatives (Miss):  {fn}")
print("-" * 30)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1_score:.4f}")

# Task 9.2: Evaluate the LSTM


### Your job

Fill in the sections in the code below marked with ...

In [ ]:
# now test the model (inference)

total_val_loss = 0.0

lstm_model.eval()


# init metrics for F1 score testing
tp = 0
fp = 0
tn = 0
fn = 0

actual_num_frames_with_throws = 0

throw_predicted_threshold = model_configurations['inference']['threshold']

successful_throw_ids = []

test_pbar = tqdm(testing_dataset_loader, desc="Testing Model", leave=True)
with torch.no_grad():
    for inputs, targets, masks, batch_ids in test_pbar:
        inputs, targets, masks = inputs.to(device), targets.to(device), masks.to(device)
        
        # run inference on the inputs
        logits = lstm_model(inputs)
        
        # turn outputs into probabilities of each frame containing a throw
        probs = torch.sigmoid(logits)



        predictions = (probs >= throw_predicted_threshold).float()

        # Only look at real frames (where mask == 1)
        # We flatten them to make the comparison easy
        active_preds = predictions[masks == 1]
        active_targets = targets[masks == 1]

        # Update counts
        tp += ((active_preds == 1) & (active_targets == 1)).sum().item()
        fp += ((active_preds == 1) & (active_targets == 0)).sum().item()
        fn += ((active_preds == 0) & (active_targets == 1)).sum().item()
        tn += ((active_preds == 0) & (active_targets == 0)).sum().item()
        

        # since only 1s and zeros, can just sum to get num frames with throws
        actual_num_frames_with_throws += active_targets.sum().item()
                
        for i in range(len(predictions)):
            if (predictions[i] == 1).any() and (targets[i] == 1).any():
                successful_throw_ids.append(batch_ids[i])
        


# hint: what are the equations for precision and recall?
precision = .../(... + ... + 1e-8)
recall = .../(actual_num_frames_with_throws + 1e-8)   # Note: we have the + 1e-8 to prevent division by 0 

f1_score = 2* (precision * recall)/(precision + recall + 1e-8)  # Note: we have the + 1e-8 to prevent division by 0 

print("----------Testing Results for LSTM----------")
print(f"Total num of frames with throws in Dataset: {actual_num_frames_with_throws}")
print(f"True Positives (Hits):   {tp}")
print(f"True Negatives:   {tn}")
print(f"False Positives (Gaps):  {fp}")
print(f"False Negatives (Miss):  {fn}")
print("-" * 30)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1_score:.4f}")

---

# Visualizing our training 📊

Often it's useful to view on a graph how our loss changes as the model trains.

You'll notice that some code was included earlier in the training loop to track the loss after every epoch and store it in a list, you can see the graph produce by this data below.


**How does the LSTM training look compared to the RNN?** 🤔

In [ ]:
import matplotlib.pyplot as plt

# plotting training
plt.figure(figsize=(10, 6))
plt.plot(train_losses_rnn, label='Training Loss', color='blue', linewidth=2)
plt.plot(eval_losses_rnn, label='Evaluation Loss', color='orange', linewidth=2, linestyle='--')
plt.title('Judo RNN Model Training Progress')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(train_losses_lstm, label='Training Loss', color='blue', linewidth=2)
plt.plot(eval_losses_lstm, label='Evaluation Loss', color='orange', linewidth=2, linestyle='--')
plt.title('Judo LSTM Model Training Progress')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



# Understanding the graphs given 👀

To interpret the training progress graphs in your notebook, you should look at the relationship between the Training Loss (solid blue line) and the Evaluation Loss (orange dashed line). 

A healthy graph, like your LSTM result, shows both lines descending smoothly together, which indicates the model is successfully generalizing — i.e. the model is learning the actual physics of Judo rather than just memorizing the training clips.

In contrast, the RNN graph shows a flatline or erratic spikes, which is a textbook sign of Exploding Gradients where the model's mathematical memory fails, causing it to stop learning and simply guess a safe baseline for every frame. 

If the orange line were to start trending upward while the blue line continues to fall, it would signal overfitting, meaning the model has reached its brain capacity and is starting to cheat by memorizing specific datapoints and their labels

License

Copyright 2026 [Conor Ryan O'Shea]

Licensed under the Apache License, Version 2.0; you may not use this file except in compliance with the License. You may obtain a copy of the License at:
http://www.apache.org/licenses/LICENSE-2.0